## 13-function-calling

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [9]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Usually, yes — if the course is still open, you can join it.\n\nA few things to check:\n- **Enrollment status:** Is registration still open?\n- **Prerequisites:** Does the course require prior approval, placement, or prerequisites?\n- **Capacity:** Are there any seats left?\n- **Start date:** If it has already started, late enrollment may or may not be allowed.\n\nIf you want, I can help you draft a short message to the instructor or coordinator asking to join.'

In [10]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [11]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

The `description` is the most important field, because the model reads it to decide when to call the function. `parameters` is a JSON schema for the arguments, and we mark `query` as required so the model always fills it in

In [12]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join it enrollment open registration new student"}', call_id='call_cZhSG4FqXY2qLm1TsQgvtKGg', name='search', type='function_call', id='fc_057539f3b0ad425b006a39a4c985608192b1f4dd40f2bd1f3a', namespace=None, status='completed')]

In [13]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

Now we send this result back to the model. First, we add the model's output to the conversation history - the model needs to see its own function call. Then we add the tool result

In [21]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

The `call_id` links the tool result to the specific function call the model requested. If the model makes multiple function calls in one turn, each one gets its own `call_id`.

In [24]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join it enrollment open registration new student"}', call_id='call_cZhSG4FqXY2qLm1TsQgvtKGg', name='search', type='function_call', id='fc_057539f3b0ad425b006a39a4c985608192b1f4dd40f2bd1f3a', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_cZhSG4FqXY2qLm1TsQgvtKGg',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "1e829a8c6f",\n    "course": "llm-zoomcamp",\n    "section": "Module 2: Vector Search",\n    "question": "Do I need a new GitHub repo for Module 2, or just a new codespace?",\n 

**Asking the model again**

We call the API a second time with the expanded history:

In [25]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join.\n\nAccording to the course FAQ, you’re accepted and can start learning and submitting homework while the form is open, even if you just discovered the course. If you want a certificate, make sure to submit your project before submissions close.'

In [26]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(782, 57)

In [27]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


## 14-agentic-loop

### Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the `developer` message. The better the instructions, the better the agent helps.
- Tools, the functions the agent can call to carry out the task. For us that's only `search`.
- Memory, the message history. We append every prompt, every model output, and every tool result. The agent reads this to know what it has already tried.

Everything below is the code that wires these three together inside a loop.

### A developer prompt

So far we've relied on the model to figure out when to search. We make that more reliable with a developer message that spells out how to behave. This is where we give the agent its role. The same message also pushes it toward multiple searches, so we get to watch the loop run more than once.

In [28]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

### A function-call helper

We'll be running function calls repeatedly inside the loop, so let's wrap that in a small helper. It turns the JSON arguments into a Python dict, calls the right function, and serializes the result. We only have one tool for now, so we dispatch on the function name directly.

In [30]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

The helper returns the exact structure the Responses API expects. When we add more tools later, we'll extend this with more `if` branches (or switch to a registry).

### Processing one response

Let's process a single model response. We append each output entry to the conversation, print any messages, and run any function calls. Function-call results get appended too.

In [31]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course enrollment discovered the course can I join"}


The has_function_calls flag tells us whether the model needs another API call. If the response contains a function call, the updated messages has tool output the model hasn't seen yet. We'll need to send it back.

### The full agent loop

We wrap this in a `while` loop. The loop keeps calling the model until it returns a response without any function calls. We also keep an iteration counter so we can see how many round-trips happened.

In [32]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course. You can start learning anytime, and the materials are available.

One important note: if you want a certificate, you need to submit your project while the course is still accepting submissions.

If you want, I can also explain how the course works for self-paced vs live cohort participation.


This is the core agent loop. The model reasons about the next action. Your code performs it, and the model sees the result on the next turn. The loop stops when the model returns a final answer with no more tool calls.

We don't decide how many times the model searches. The model does, and we keep looping until it stops asking for tools.

The exit condition is the simplest one possible. No function calls this turn means we're done. Other frameworks add safety nets on top, like a max iteration count, a token budget, or a wall-clock limit. You might cap it at five iterations and force an answer on the last one. The core is still this one flag.

### Wrapping it in a function

Let's wrap the loop in a function so we can reuse it. The function takes the instructions and the question as parameters, and returns the final answer.

In [34]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [35]:
# Try it with a question that has a typo:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install setup local model course FAQ"}
function_call: search {"query":"Ollama locally run install local development FAQ"}
iteration #2...
ASSISTANT:
To run **Ollama locally**, do this:

1. **Install Ollama**
   - macOS: download the `.pkg` from [ollama.com/download](https://ollama.com/download)
   - Windows: download the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model**
   ```bash
   ollama run llama3
   ```
   This will download the model and open a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response showing available models.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(response['

'To run **Ollama locally**, do this:\n\n1. **Install Ollama**\n   - macOS: download the `.pkg` from [ollama.com/download](https://ollama.com/download)\n   - Windows: download the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and open a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response showing available models.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you’re getting a connection issue, restart the Ollama server with:\n```bash\nollama serve\n```\n\nOr in a notebook:\n```bash\n!nohup oll

Watch what happens. The agent searches for "Olama" and gets poor results. It then searches again with "Ollama" and finds the answer. The loop lets the model recover from a bad search on its own. That's the whole point of going agentic.

In [36]:
# Also try the course enrollment question:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered the course still join enroll add drop late registration"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, just make sure you submit your project while submissions are still open.

If you'd like, I can also help you with how to start the course, where to find the materials, or what the usual weekly workflow looks like.


"Yes — you can still join the course.\n\nIf you want a certificate, just make sure you submit your project while submissions are still open.\n\nIf you'd like, I can also help you with how to start the course, where to find the materials, or what the usual weekly workflow looks like."

### Encouraging multiple searches

There's a subtle issue here. The model often answers after the first search, even when more searches would help. It reasons that it already knows enough, so why bother. We push it to explore more by rewriting the instructions.

In [37]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course after discovering late enrollment can I join the course FAQ"}
iteration #2...
function_call: search {"query":"certificate project submit while accepting submissions live cohort peer review FAQ join course self-paced"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If your goal is a certificate, make sure you submit your project while submissions are still open, because certificates are only available for the live cohort process.

If you want, I can also help with how to get started or explain the certificate requirements. Is there anything else you’d like to explore?


'Yes — you can still join the course.\n\nIf your goal is a certificate, make sure you submit your project while submissions are still open, because certificates are only available for the live cohort process.\n\nIf you want, I can also help with how to get started or explain the certificate requirements. Is there anything else you’d like to explore?'

Now the agent makes multiple searches per question and doesn't stop after the first round of results. The instructions are how we steer the agent. It can still decide to skip ahead sometimes, so don't expect it to follow them every single run.

### Restricting off-topic questions

Right now the agent will answer anything. Ask it about chess and it will still try.

In [38]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit what is queen gambit"}
iteration #2...
function_call: search {"query":"queen gambit chess opening explanation queen's gambit definition"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a classic **chess opening**.

It starts with:
1. **d4 d5**
2. **c4**

White offers the **c-pawn** as a gambit to try to draw Black’s **d-pawn** away from the center and gain stronger control of the board.

A few key points:
- It’s called a **gambit** because one side offers a pawn for positional advantage.
- It’s one of the oldest and most respected openings in chess.
- Black can accept or decline the gambit.

If you meant something else by “queen gambit,” tell me and I can clarify. Would you like a simple explanation of the main lines too?


'The **Queen’s Gambit** is a classic **chess opening**.\n\nIt starts with:\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** as a gambit to try to draw Black’s **d-pawn** away from the center and gain stronger control of the board.\n\nA few key points:\n- It’s called a **gambit** because one side offers a pawn for positional advantage.\n- It’s one of the oldest and most respected openings in chess.\n- Black can accept or decline the gambit.\n\nIf you meant something else by “queen gambit,” tell me and I can clarify. Would you like a simple explanation of the main lines too?'

We want a course assistant, not a general chatbot. We tighten the instructions so the agent only answers from the FAQ. For our own use we might be fine letting it answer from general knowledge. So treat this mainly as an illustration of steering scope.

In [39]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit"}
iteration #2...
function_call: search {"query":"queen gambit queen's gambit course faq chess opening"}
iteration #3...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen gambit,” and it looks like this is probably a chess question rather than a course/logistics question.

If you meant something else course-related, feel free to rephrase it, and I can check the FAQ again. Is there another area you want to explore?


'I couldn’t find a course FAQ entry for “queen gambit,” and it looks like this is probably a chess question rather than a course/logistics question.\n\nIf you meant something else course-related, feel free to rephrase it, and I can check the FAQ again. Is there another area you want to explore?'

This is a lightweight form of an input guardrail. We tell the agent what's in scope and what isn't. A real guardrail checks the input before the agent runs and can block off-topic questions outright. That's a separate topic, but instructions are the first place to start.

This handwritten loop is the best way to understand what frameworks hide from you. Every agent framework wraps this same pattern, whether it's LangChain, PydanticAI, or the OpenAI Agents SDK.

## 15-frameworks (ToyAIKit)

The handwritten agent loop from the previous lesson is educational but
repetitive. Every time you build a new agent, you'd write the same
while-loop, the same function-call handling, the same message
management.

ToyAIKit wraps this pattern so you can focus on tools, prompts, and
behavior. We built it together in a DataTalks.Club workshop a while
back. It does the same thing as our handwritten loop with less
boilerplate. If you open its `runners` code, you'll find the same
`while True` loop we wrote by hand.

I use it here on purpose, because I don't want to pick a winner among
the production frameworks. ToyAIKit is small and easy to read, so when
something breaks you can see exactly what happened. That makes it handy
for developing and debugging locally before you go to production.

One caveat. ToyAIKit is a teaching and experimentation library, and it
is NOT meant for production use. We use it because it's minimal and you
can see what it does.

In [40]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [41]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

### Letting ToyAIKit generate the schema

Writing that schema by hand is annoying, and we don't want to do it
for every function. So we don't have to.

If we add a type hint and a docstring to `search`, ToyAIKit reads them
and derives the schema for us:

In [43]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

Then register it without passing a schema:

In [44]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [46]:
# You can look at what ToyAIKit produced.
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

The output is the same JSON schema we hand-wrote in the function calling lesson. ToyAIKit generated it from the docstring and the type hint.

Every modern agent framework does this same trick. It reads a typed Python function with a docstring and builds the schema from it. The OpenAI Agents SDK, PydanticAI, LangChain and Google ADK all work this way. You write the tool and the framework figures out how to describe it.

### The chat interface and runner

Create the chat interface and a callback, then build the runner:

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

The `chat_interface` handles display in the notebook. The `callback`
renders model messages and tool calls as they happen. The runner runs
the agent loop, the same `while True` we wrote by hand. It sends
messages, executes function calls, adds tool outputs back, and repeats
until the model is done.

We pick `gpt-5.4-mini` here on purpose. Without it, ToyAIKit falls
back to a smaller, faster default that doesn't follow the instructions
as reliably.

In [49]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


We used the typo "Olama" on purpose. The agent searches and gets poor
results, then retries with "Ollama". The recovery is the same as the
handwritten loop. The notebook output is nicer to watch. Each tool
call and message renders inline, so you can look at every search
result.

The `result` is a `LoopResult` with `all_messages` (the full
conversation), token counts, and `cost` (computed from token usage).

### Cost and tokens

In [ ]:
#Look at what the call cost:
result.cost

CostInfo(input_cost=Decimal('0.00111075'), output_cost=Decimal('0.001278'), total_cost=Decimal('0.00238875'))

Useful while developing - especially with multi-turn agents where one prompt can trigger several model calls. The handwritten loop made you compute this by hand. The framework keeps a running total for you.

In [51]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run Ollama local installation run locally"}', ca

This is just a list - the same `messages` list we maintained by hand.

### Continuing the conversation

Take the messages from the previous result and pass them as
`previous_messages` on the next `loop` call:

In [52]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


The runner picks up where the last call left off, with the same agent loop and an extended history. The model knows "different model" refers to Ollama because it sees the previous turn in memory. Without that history, it would have no idea what we're asking about.

### Interactive chat

For a chat-like workflow, run the built-in input loop:

In [ ]:
# runner.run()

-> Response received


-> Response received


-> Response received


Type questions and get answers. Type "stop" to exit.

## 16-other-frameworks

Video: [Watch this lesson](https://www.youtube.com/watch?v=4yiCbKX9RhI&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

The concepts you learned in Part 2 are the same across frameworks.
Function calling, the agent loop, and tool definitions all wrap the
same pattern. Send messages, run any function calls, and repeat until
the model is done.

You now understand how the loop works. So you can pick up any
production framework and know what it's doing under the hood. I kept
this module framework agnostic on purpose, so you can explore and pick
the one you like.

Here are some frameworks worth a look:

### OpenAI Agents SDK

The official SDK from OpenAI for building agents. It uses the same
Responses API we used throughout this module. It supports tool
definition, multi-turn conversations, and handoffs between agents.

```bash
uv add openai-agents
```

Good choice if you're already using OpenAI and want something
official and well-maintained.

### PydanticAI

A type-safe agent framework that supports multiple LLM providers.
Tools are plain Python functions with type hints, no wrappers needed.
Switching providers is as simple as changing the model string.

```bash
uv add pydantic-ai
```

This is my personal favorite. The reason isn't the type safety, since
the other frameworks have that too. It's how it feels to use, and the
team behind Pydantic. Good choice if you want type safety and
multi-provider support.

### LangChain / LangGraph

A popular framework with lots of integrations. LangChain handles the
basics, and LangGraph adds graph-based workflows for more complex
agent patterns.

Good choice if you need lots of integrations (vector stores, document
loaders, etc.) and a large community.

### Google ADK

The Agent Development Kit from Google. If you plan to use Gemini
models, this is the one I'd reach for. It exposes the same building
blocks we've seen, like tools, instructions, and sessions. It also
integrates with Google Cloud.

Good choice if your stack is on Google Cloud or you specifically want
Gemini.

### Others

Other frameworks worth knowing:

- CrewAI - multi-agent orchestration
- AutoGen - multi-agent conversations from Microsoft
- Semantic Kernel - from Microsoft, supports C# and Python
- Smolagents - lightweight agent framework from HuggingFace
- Anthropic Tool Use - Anthropic's native tool use API

Pick one that fits your stack and your needs. The hard part is
designing good tools and prompts - the loop is always the same.

### Avoiding agents when you can

One closing thought. You don't always need an agent, and that's fine.
You might assume your app needs one, but often it doesn't.

Adding an agent is a real cost:

- More API calls per request, since the loop can fire many tool calls
  before the model is satisfied.
- Higher latency, since each round-trip waits on the model.
- More money spent, since every iteration is another billed call and
  the full message history is re-sent each turn.
- More moving parts, since you now monitor cost, iteration count, and
  whether the agent is going in circles.
- Less predictable behavior, since the LLM decides what to do and two
  runs of the same prompt can take different paths.

Before reaching for an agent, ask whether the problem needs one. A lot
of tasks are well served by simpler approaches.

Reach for one of these first:

- plain RAG, with one search and one answer
- parsing or templating a document into another form
- a single LLM call with no tools

Try the simpler approach first, and if it works, ship it. Reach for
the agent loop only when you've tried the simpler solution and it
genuinely can't handle the problem. By then you'll know the extra
complexity is worth it, and you'll be ready to take it on.
